(mmm_cost_per_unit)=

# Cost-per-Unit: Model What Drives Revenue, Report What You Spend

Imagine you are a marketing analyst at an e-commerce company. You run two
advertising channels — say a social-media channel (measured in **impressions**)
and a TV channel (measured in **dollars**).

Your data scientist fits a Media Mix Model on this data. The model learns how
each channel's *input* drives *revenue*. But there is a catch: **the two channels
are measured in completely different units.** Comparing their returns side-by-side
is meaningless unless we first bring them onto a common scale.

## When does this matter?

The impressions-vs-dollars scenario above is just one example. In practice,
channels can differ in unit for many reasons:

- **Impressions, clicks, or other engagement metrics** — any non-monetary media
  input whose price fluctuates over time.
- **Different currencies** — e.g. one channel invoiced in USD and another in EUR.
  Exchange rates move, so pre-converting to a single currency before fitting
  bakes FX noise into the predictor.
- **Any heterogeneous unit** — the core idea is the same whenever the raw media
  input is not directly in the currency you use for reporting.

## Why not just convert everything to dollars upfront?

One obvious solution is to convert every channel to a common dollar unit before
fitting — multiply impressions by the cost-per-impression, convert EUR to USD,
etc. Simple enough.

But this approach has a serious flaw: **cost-per-unit conversions are not fixed.**
The price of an impression fluctuates with auction dynamics, seasonality,
competitor activity, and platform pricing changes. Exchange rates move daily.
Converting before modelling bakes those price fluctuations directly into the
predictor variable, adding noise that has nothing to do with the media's actual
effect on consumers.

When impressions are the causal driver — a consumer *sees* an ad — modelling
on impressions keeps the signal clean and separates the *media effect* from the
*media cost*.

## The approach: model in native units, convert at reporting time

This is exactly what **`cost_per_unit`** enables. We fit the model on whatever
unit best represents each channel's causal input (impressions, clicks, local
currency, etc.) and supply the cost-per-unit separately as a post-fit parameter.
This means:

- The model estimates the effect in the channel's native unit — preserving the
  true causal relationship.
- At reporting and optimisation time, we apply the cost-per-unit to translate
  everything into a common currency for fair channel comparison.
- We can update the cost-per-unit assumption without re-fitting the model,
  making it easy to explore how pricing or exchange-rate changes affect budget
  recommendations.

In this notebook we will:

1. **Fit** an MMM on the raw data (impressions + dollars)
2. **Inspect** ROAS and saturation curves *before* any unit conversion — and see
   how misleading they can be
3. **Set `cost_per_unit`** for the social-media channel (\$0.10/impression) and
   watch the story change dramatically
4. **Run budget optimization** under different cost-per-impression assumptions
   (\$0.05, \$0.10, \$0.15) to see how pricing affects allocation

## A note on choosing the right unit

That said, **there is no universal rule that impressions are always better than
spend.** The right choice depends on data quality and the modelling context.
If your impression data is unreliable or noisy (e.g. inconsistent tracking
across platforms), modelling on spend may actually produce a cleaner signal.
It is up to the modeller to decide which unit best represents the true causal
input for each channel. `cost_per_unit` supports either direction — you can
model on impressions and convert to dollars, or model on spend and convert back.

## Step 1: Load Data and Fit the Model

We use a two-channel dataset from `multidimensional_mock_data.csv`, filtered to
`geo_a`. The model is built from a YAML config with `GeometricAdstock` and
`MichaelisMentenSaturation`.

In [1]:
%load_ext autoreload
%autoreload 2

import warnings

import arviz as az
import numpy as np
import pandas as pd
import plotly.io as pio
import xarray as xr

from pymc_marketing.mmm import PowerPriceResponse
from pymc_marketing.mmm.builders.yaml import build_mmm_from_yaml
from pymc_marketing.mmm.mmm import (
    BudgetOptimizerWrapper,
)
from pymc_marketing.paths import data_dir

warnings.filterwarnings("ignore")
pio.renderers.default = "notebook_connected"

az.style.use("arviz-darkgrid")


df = pd.read_csv(data_dir / "multidimensional_mock_data.csv", parse_dates=["date"])
df_geo_a = df.loc[df["geo"] == "geo_a"].reset_index(drop=True)
x_train = df_geo_a[["date", "x1", "x2"]].rename(columns={"x1": "Social", "x2": "TV"})
# Scale Social to simulate impression-level values (the raw mock data uses a
# smaller scale; multiplying by 10 gives realistic impression counts).
x_train["Social"] = x_train["Social"] * 10
y_train = df_geo_a["y"]
x_train.head()

,date,Social,TV
0,2018-04-02,1592.900090,0.0
1,2018-04-09,561.942382,0.0
2,2018-04-16,1462.001331,0.0
3,2018-04-23,356.992763,0.0
4,2018-04-30,1933.725768,0.0


In [2]:
# We use only 2 chains for speed — divergences and convergence warnings
# are expected and harmless for the purposes of this demo.
seed: int = sum(map(ord, "cost_per_unit"))
mmm = build_mmm_from_yaml(
    X=x_train,
    y=y_train,
    config_path=data_dir / "config_files" / "cost_per_unit_example.yml",
    model_kwargs={
        "sampler_config": {
            "chains": 2,
            "tune": 1000,
            "draws": 1000,
            "random_seed": seed,
        }
    },
)
_ = mmm.fit(x_train, y_train)

NUTS[nutpie]: [intercept_contribution, adstock_alpha, saturation_lam, saturation_alpha, y_sigma]


Output()

Output()

## Step 2: The Misleading Picture — ROAS Without Unit Conversion

Let's look at the fitted ROAS and saturation curves *as-is*, without any
cost-per-unit adjustment. Remember:

- **Social** is in **impressions**
- **TV** is in **dollars**

So the "ROAS" for Social is really *revenue per impression*, while TV's
ROAS is a proper *revenue per dollar*. Comparing them side-by-side is like
comparing apples and oranges. Without cost-per-unit adjustment, Social's
apparent ROAS will appear roughly **10x smaller** than TV's — not because
Social is less effective, but because it is being measured in impressions
rather than dollars.

In [3]:
fig_roas_before = mmm.plot_interactive.roas()
fig_roas_before

**Notice how TV dominates.** Its ROAS is expressed in \$/\$ and
looks roughly **10x larger** than Social's revenue-per-impression
metric — exactly as expected when one channel is measured in impressions and
the other in dollars.

A naive reading of this chart would suggest: *"TV is far more efficient —
pour all the money there."*

Let's look at the saturation curves next. The x-axis for each channel is in
its *native unit*, so the scales are not comparable.

In [4]:
mmm.plot_interactive.saturation_curves()

Sampling: []


Output()

The saturation curves above look reasonable individually, but they are **not comparable** — Social Media's x-axis is *impressions* while TV's is *dollars*. We cannot visually judge which channel gives more bang for the buck.

---

## Step 3: Setting `cost_per_unit` — Leveling the Playing Field

Now we tell the model what Social Media impressions actually *cost*. By setting
`cost_per_unit = \$0.10` for Social Media, we bridge the gap from impressions
back to dollars. The model can then express everything in consistent **\$/\$**
terms.

- **`channel_1` (Social Media):** `cost_per_unit = 0.1` (\$0.10 per impression)
- **`channel_2` (TV):** `cost_per_unit = 1.0` (already in dollars — this is the default when omitted)

There are **two ways** to attach `cost_per_unit` to a model:

1. **At initialization** — pass the DataFrame directly to the `MMM()`
   constructor via the `cost_per_unit` parameter. The conversion factors are
   then applied automatically after `fit()`.
2. **Post-hoc** — call `mmm.set_cost_per_unit(df)` on an already-fitted model,
   which is what we do below. This is useful when you want to experiment with
   different cost assumptions without refitting.

In [5]:
dates = mmm.data.dates
cost_per_unit_df = pd.DataFrame(
    {
        "date": dates,
        "Social": np.ones(len(dates)) * 0.1,
    }
)
cost_per_unit_df.head(10)

,date,Social
0,2018-04-02,0.1
1,2018-04-09,0.1
2,2018-04-16,0.1
3,2018-04-23,0.1
4,2018-04-30,0.1
5,2018-05-07,0.1
6,2018-05-14,0.1
7,2018-05-21,0.1
8,2018-05-28,0.1
9,2018-06-04,0.1


In [6]:
# and now let's set it
mmm.set_cost_per_unit(cost_per_unit_df)

### Verifying the Conversion

`get_channel_data()` returns the **raw** values (impressions for Social Media,
dollars for TV). `get_channel_spend()` multiplies by `cost_per_unit`, so now
**both** columns are in dollars.

In [7]:
raw = mmm.data.get_channel_data()
spend = mmm.data.get_channel_spend()
raw_df = (
    raw.isel(date=slice(0, 10)).to_dataframe().unstack("channel").add_suffix("_raw")
)
spend_df = (
    spend.isel(date=slice(0, 10))
    .to_dataframe()
    .unstack("channel")
    .add_suffix("_spend($)")
)
comparison = pd.concat([raw_df, spend_df], axis=1)
comparison

channel_data_raw             channel_spend_spend($)            
channel          Social_raw      TV_raw        Social_spend($) TV_spend($)
date                                                                      
2018-04-02      1592.900090    0.000000             159.290009    0.000000
2018-04-09       561.942382    0.000000              56.194238    0.000000
2018-04-16      1462.001331    0.000000             146.200133    0.000000
2018-04-23       356.992763    0.000000              35.699276    0.000000
2018-04-30      1933.725768    0.000000             193.372577    0.000000
2018-05-07       235.854931    0.000000              23.585493    0.000000
2018-05-14      2121.245524    0.000000             212.124552    0.000000
2018-05-21      1669.600873  439.890918             166.960087  439.890918
2018-05-28      1265.351424    0.000000             126.535142    0.000000
2018-06-04      4690.272079    0.000000             469.027208    0.000000

## Step 4: The True Picture — ROAS *After* Cost-per-Unit

Now that both channels are in consistent dollar terms, let's revisit the ROAS
chart.

In [8]:
fig_roas_after = mmm.plot_interactive.roas()
fig_roas_after

**The story has changed dramatically.** Social Media's ROAS has jumped —
because dividing revenue by *actual dollar spend* (impressions × \$0.10)
instead of raw impression counts gives a much larger number. The two
channels are now **much more comparable**.

The takeaway: TV is *not* overwhelmingly better. Social Media delivers
competitive returns when measured properly.

Let's confirm this with the saturation curves, now plotted with a consistent
**dollar (Spend)** x-axis.

In [9]:
mmm.plot_interactive.saturation_curves(max_value=3)

Sampling: []


Output()

Now both saturation curves share the same unit on the x-axis — **dollars
spent**. You can directly compare the marginal return of an extra dollar on
Social Media vs. TV, which is exactly what you need for budget planning.

---

## Step 5: Budget Optimization with `cost_per_unit`

For budget optimization, we provide a *future* `cost_per_unit` for the
planning window. This is independent of the historical values — impression
prices may change over time.

The optimizer:
1. Takes a **total dollar budget**
2. Allocates dollars across channels and time periods
3. Converts Social Media's dollar allocation to *impressions* using
   `cost_per_unit` before evaluating the response function
4. Returns optimal budgets **in dollars**

In [10]:
num_periods = 4
freq = pd.infer_freq(mmm.data.dates)
future_dates = pd.date_range(
    start=mmm.data.dates[-1] + pd.tseries.frequencies.to_offset(freq),
    periods=num_periods,
    freq=freq,
)
budget_wrapper = BudgetOptimizerWrapper(
    model=mmm,
    start_date=future_dates[0],
    end_date=future_dates[-1],
)

### Sensitivity Analysis: How Impression Price Affects Allocation

The cost per impression is not fixed — it varies with market conditions,
bidding strategy, and seasonality. Let's see how the optimal budget
allocation shifts when Social Media impressions cost **\$0.05**, **\$0.10**,
or **\$0.15** each.

- **Cheaper impressions (\$0.05)** → more impressions per dollar → Social
  Media becomes more attractive
- **Baseline (\$0.10)** → our best estimate of current pricing
- **Expensive impressions (\$0.15)** → fewer impressions per dollar → budget
  shifts toward TV

In [11]:
def compare_budgets_optimization_for_different_cpus(budget):
    rates = [0.05, 0.1, 0.15]
    results_list = []
    for rate in rates:
        cpu_df = pd.DataFrame(
            {
                "date": future_dates,
                "Social": [rate] * num_periods,
            }
        )
        budgets, _ = budget_wrapper.optimize_budget(
            budget=budget,
            cost_per_unit=cpu_df,
        )
        row = budgets.to_dataframe(name="budget").reset_index()
        row["cost_per_impression"] = f"${rate:.2f}"
        results_list.append(row)

    comparison_df = pd.concat(results_list)
    return comparison_df.pivot_table(
        index="cost_per_impression",
        columns="channel",
        values="budget",
        aggfunc="sum",
    )


compare_budgets_optimization_for_different_cpus(1000)

channel,Social,TV
cost_per_impression,,
$0.05,476.154845,523.845155
$0.10,452.070717,547.929283
$0.15,410.293529,589.706471


### Reading the Results (Small Budget)

At \$1,000 total budget, the optimizer behaves intuitively: as the cost per
impression **decreases** (from \$0.15 → \$0.05), each dollar buys more Social
Media impressions, making that channel more efficient. The optimizer responds
by shifting more budget toward Social Media.

But does this pattern hold at every budget level? Let's run the same
comparison with **10× the budget** to see:

In [12]:
compare_budgets_optimization_for_different_cpus(10_000)

channel,Social,TV
cost_per_impression,,
$0.05,3976.436582,6023.563418
$0.10,4211.632120,5788.367880
$0.15,4355.290385,5644.709615


### Why the Pattern Reverses at Higher Budgets

The \$10,000 result is **the opposite** of the \$1,000 case: as impressions get
*cheaper*, Social actually receives *less* dollar budget. This is
counter-intuitive but mathematically correct — it is driven by **saturation**.

At a \$1,000 budget the channels still have room to grow: cheaper impressions
mean more impressions per dollar, so the channel with the lower
cost-per-unit is genuinely more *efficient*. The optimizer rewards that
efficiency by shifting budget toward Social.

At \$10,000, both channels are deep into their saturation curves. Once a
channel is saturated, additional spend yields almost no incremental response.
The optimizer's job reduces to distributing dollars so that each channel
*just* reaches its plateau. When impressions are cheap, fewer dollars are
needed to buy enough impressions to saturate Social — so the remaining
dollars spill over to TV. When impressions are expensive, the optimizer must
allocate *more* dollars to Social to purchase the same number of impressions.

In short, at low budgets we are in the **efficiency regime** (cheaper =
higher return per dollar → more Social), while at high budgets we enter the
**saturation regime** (cheaper = Social saturates with fewer dollars →
surplus flows elsewhere).

> **Practical note:** A scenario where *every* channel is deeply saturated is
> a signal that your budget exceeds the capacity of your current media mix.
> In practice, rather than pouring money into channels that have already
> plateaued, this is the point where you should consider **developing new
> channels** — finding untapped audiences or formats that still offer
> meaningful marginal returns.

This is exactly the kind of scenario analysis that `cost_per_unit` enables:
same model, same posterior — but different economic assumptions about media
pricing lead to different optimal strategies.

(mmm_cost_per_unit_price_response)=
## Step 6: When the Price Rises With the Spend

So far each impression costs the same, whatever the plan buys. Auction inventory does not work like that: buying more in a week bids the price up. {class}`~pymc_marketing.mmm.price_response.PowerPriceResponse` gives the budget optimizer a unit price that rises as a power of the money spent in the period:

$$
p(s) = p_0 \left(\frac{s}{s^{\text{ref}}}\right)^{\gamma}, \qquad 0 \le \gamma < 1
$$

$p_0$ is the `cost_per_unit` passed to the optimizer. $s^{\text{ref}}$ is the per-period spend at which that price applies (`reference_spend`). $\gamma$ is the price elasticity. At $\gamma = 0$ the price is constant, which is the optimizer of Step 5.

This section uses {meth}`~pymc_marketing.mmm.mmm.MMM.budget_optimizer`, which takes the window `cost_per_unit` as an `xarray.DataArray` over `(date, channel)`. TV is in dollars, so its price per unit is 1.

In [13]:
window_start, window_end = future_dates[0], future_dates[-1]


def window_cost_per_unit(social_rate: float) -> xr.DataArray:
    """Price per unit over the window: dollars per impression for Social, 1 for TV."""
    return xr.DataArray(
        np.tile([social_rate, 1.0], (num_periods, 1)),
        dims=("date", "channel"),
        coords={"date": future_dates, "channel": ["Social", "TV"]},
    )


def price_optimizer(price_response, social_rate: float = 0.1):
    """Budget optimizer over the 4-week window with the given price response."""
    return mmm.budget_optimizer(
        window_start,
        window_end,
        cost_per_unit=window_cost_per_unit(social_rate),
        price_response=price_response,
    )

### The precondition: the fit must be in delivery units

A curved price map is only sound when the model was fitted on delivery units (impressions, clicks) or on spend deflated to constant prices. A saturation curve fitted on nominal spend has already absorbed the curvature of the price, so a concave price map on top would bend it twice. Here Social is fitted on impressions, so it can be priced on a curve. TV is fitted on dollars, so it stays at $\gamma = 0$.

The fitted model cannot tell which units its data are in, so the optimizer refuses a curved response until you state it:

In [14]:
try:
    price_optimizer(PowerPriceResponse(elasticity={"Social": 0.3}))
except ValueError as error:
    print(error)

1 validation error for BudgetOptimizer
  Value error, channel_data: price_response bends the price of channels ['Social']. That is sound only if the model was fitted on delivery units (impressions, clicks) or on spend deflated to constant prices; a saturation curve fitted on nominal spend has already absorbed the price curvature, and a curved map on top would bend it twice. The fitted model cannot establish its own units (a historical cost_per_unit table is a declaration, not evidence, and can be set after the fit). If the fit is in delivery units or constant-price spend, pass assume_delivery_units=True; the historical table then anchors reference_spend on the channels it prices, and any other curved cell needs a reference_spend (per-period money, the units of result.budgets). [type=value_error, input_value={'model': <pymc.model.cor...ibution_original_scale'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error


Both ways past this check need `assume_delivery_units=True`. The historical `cost_per_unit` table of Step 3 is not proof of the units: `set_cost_per_unit` writes it after the fit, without touching the model. What the table does is anchor the price:

1. **Attest, and let the table anchor $p_0$.** Step 3 priced Social, so the optimizer derives Social's `reference_spend` from the fit.
2. **Attest, and pass `reference_spend` yourself.** This is required for any curved channel the table does not price, and it overrides the derived value channel by channel.

The rule is about curvature, not about price responses in general: a channel left at $\gamma = 0$ needs no attestation, since its money passes through unbent. Bracket rates, where the marginal price steps up at a breakpoint, are tracked in [#3067](https://github.com/pymc-labs/pymc-marketing/issues/3067).

The derived reference is the mean spend per period over the weeks the channel was on air, in the units of `total_budget`:

In [15]:
channel_spend = mmm.idata.constant_data["channel_spend"]
derived_reference = channel_spend.where(channel_spend > 0).mean("date")
social_reference = float(derived_reference.sel(channel="Social"))
mean_total_spend = float(channel_spend.sum("channel").mean())
print(f"Social reference_spend: ${social_reference:,.2f} per week")
print(f"Mean total spend:       ${mean_total_spend:,.2f} per week")

Social reference_spend: $154.54 per week
Mean total spend:       $235.47 per week


### Elasticity sweep

The model never sees prices, so $\gamma$ is an input. It comes from buying data, with its own problems. Treat it as a sensitivity sweep and run $\gamma = 0$ next to the values you believe.

The table below reports, for Social: the weekly budget, the impressions it buys per week, the average price per impression (`implied_price`), and the price of the last impression bought (`implied_marginal_price`). Except very close to zero spend, the marginal price is `implied_price / (1 - elasticity)`.

In [16]:
def elasticity_sweep(
    total_budget: float,
    social_rate: float = 0.1,
    elasticities: tuple[float, ...] = (0.0, 0.2, 0.4),
    reference_spend: xr.DataArray | None = None,
) -> pd.DataFrame:
    """Optimize the same weekly budget at several Social elasticities."""
    rows = []
    for elasticity in elasticities:
        optimizer = price_optimizer(
            PowerPriceResponse(
                elasticity={"Social": elasticity},
                assume_delivery_units=True,
                reference_spend=reference_spend,
            ),
            social_rate=social_rate,
        )
        result = optimizer.allocate_budget(total_budget=total_budget)
        social = {"channel": "Social"}
        rows.append(
            {
                "elasticity": elasticity,
                "Social budget": float(result.budgets.sel(social)),
                "TV budget": float(result.budgets.sel(channel="TV")),
                "Social impressions": float(
                    result.implied_delivery.sel(social).mean("date")
                ),
                "implied_price": float(
                    result.implied_price.sel(social).mean("date", skipna=True)
                ),
                "implied_marginal_price": float(
                    result.implied_marginal_price.sel(social).mean("date", skipna=True)
                ),
            }
        )
    return pd.DataFrame(rows).set_index("elasticity").round(3)


elasticity_sweep(1_000)

,Social budget,TV budget,Social impressions,implied_price,implied_marginal_price
elasticity,,,,,
0.0,441.553,558.447,4415.534,0.100,0.100
0.2,348.881,651.119,2964.492,0.118,0.147
0.4,250.133,749.867,2063.099,0.121,0.202


At \$1,000 per week, Social is bought above its reference, so a larger $\gamma$ makes its impressions dearer. The marginal price rises faster than the average price, and the optimizer allocates on the marginal price, so money moves from Social to TV.

This is also why the optimizer does not solve at a fixed price and then reprice. That loop converges to $R'(u_i) / p_i = \text{const}$, but the true first-order condition is $R'(u_i)(1 - \gamma_i) / p_i = \text{const}$. With one common $\gamma$ the factor cancels. When elasticities differ across channels, the loop gives too much to the high-elasticity channels.

### Below the reference, units get cheaper

The power law is as confident below the reference as above it. Cut the weekly budget below the historical spend, and Social is bought below its reference, so its price falls under $p_0 = 0.10$:

In [17]:
below_reference = elasticity_sweep(100)
below_reference

,Social budget,TV budget,Social impressions,implied_price,implied_marginal_price
elasticity,,,,,
0.0,65.121,34.879,651.210,0.100,0.100
0.2,55.292,44.708,679.112,0.081,0.102
0.4,45.437,54.563,741.418,0.061,0.102


Social now gets less money but buys more impressions as $\gamma$ grows. With a total budget below the historical spend, every priced channel typically reports a price under $p_0$. That is the usual planning case when budgets are cut.

This can also move money *toward* a channel. Raise Social's base price to \$0.75 per impression. At a constant price, Social is not worth buying. Once it is priced, its first dollars buy impressions at a fraction of $p_0$:

In [18]:
elasticity_sweep(1_000, social_rate=0.75)

,Social budget,TV budget,Social impressions,implied_price,implied_marginal_price
elasticity,,,,,
0.0,0.000,1000.000,0.000,NaN,NaN
0.2,8.971,991.029,21.136,0.424,0.531
0.4,16.992,983.008,54.790,0.310,0.517


At $\gamma = 0$ nothing is bought, so there is no price to report (`NaN`). Here the more elastic channel receives more, not less. Real auction inventory is often not symmetric like this: floor prices and minimum bids hold the price up below the reference. So:

- anchor `reference_spend` at the level you plan to buy at, not only where $p_0$ was observed (raise `reference_spend_tolerance` when that is far from the fitted spend),
- read `implied_price` on every channel before trusting the allocation,
- sweep `elasticity` rather than fix one value.

A variant that stays flat below the reference is tracked in [#3089](https://github.com/pymc-labs/pymc-marketing/issues/3089).

Here is the \$100 budget again, with Social's `reference_spend` anchored at the spend the constant-price plan buys there, instead of at the historical mean:

In [19]:
planned_social = xr.DataArray(
    [below_reference.loc[0.0, "Social budget"]],
    dims="channel",
    coords={"channel": ["Social"]},
)
elasticity_sweep(100, reference_spend=planned_social)

,Social budget,TV budget,Social impressions,implied_price,implied_marginal_price
elasticity,,,,,
0.0,65.121,34.879,651.210,0.100,0.100
0.2,42.823,57.177,465.681,0.092,0.115
0.4,29.860,70.140,407.893,0.073,0.122


Anchored where the plan buys, Social's marginal price is above $p_0$ again, so a larger $\gamma$ moves money from Social to TV, as at \$1,000. With the historical anchor, the elastic plans bought more impressions than the $\gamma = 0$ plan. Anchored at the planned spend, they buy fewer. The anchor alone flips the answer, which is why it belongs at the level you plan to buy at.

### `reference_spend` is money per period

`reference_spend` is in the units of `total_budget` and `result.budgets`: money per period. A common mistake is to pass a window total, such as a quarter's spend, as if it were a weekly rate. That shifts every price by `num_periods ** elasticity`. So the optimizer refuses a supplied value that is more than `reference_spend_tolerance` (default 10) times away from the derived one:

In [20]:
quarter_total = 13 * derived_reference.sel(channel=["Social"])
try:
    price_optimizer(
        PowerPriceResponse(
            elasticity={"Social": 0.3},
            assume_delivery_units=True,
            reference_spend=quarter_total,
        )
    )
except ValueError as error:
    print(error)

1 validation error for BudgetOptimizer
  Value error, channel_data: price_response: reference_spend at cell ('Social',) is 2009, but the fitted spend's on-air mean per period is 154.5 (13x apart; tolerance 10x). reference_spend is per-period money, the units of result.budgets and total_budget -- a value summed over the window is off by num_periods and shifts every price by num_periods ** elasticity. Fix the units, or raise reference_spend_tolerance if the difference is intended. [type=value_error, input_value={'model': <pymc.model.cor...ibution_original_scale'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error


A total over this 4-week window is only 4 times away, so it passes the tolerance. The optimizer still names the likely mistake in a warning (this notebook silences warnings, so the cell records it explicitly). For short windows, lower `reference_spend_tolerance` if that is a risk.

In [21]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    price_optimizer(
        PowerPriceResponse(
            elasticity={"Social": 0.3},
            assume_delivery_units=True,
            reference_spend=num_periods * derived_reference.sel(channel=["Social"]),
        )
    )
for warning in caught:
    if "window total" in str(warning.message):
        print(warning.message)

channel_data: price_response: reference_spend is num_periods (4) times the fitted per-period spend on every cell it can be checked on, to within 5%: it looks like a window total. reference_spend is per-period money, the units of result.budgets and total_budget -- if so, divide by 4.


The same unit rule explains the window length. `total_budget` is per period, so a given weekly plan pays the same price relative to $p_0$ over any window. If the *window total* is held fixed instead, a shorter window spends more per week and pays more.

### Scoring a plan

`result.implied_delivery` holds the impressions each plan buys per week, which is what the model sees. To score a plan, pass its budgets to {meth}`~pymc_marketing.mmm.budget_optimizer.BudgetOptimizer.evaluate_response_distribution`. It runs the same graph as the solver, price map included, and returns the posterior of the response.

Below, both plans are scored at $\gamma = 0.4$: the plan from a constant price, and the plan that accounts for the price rise.

In [22]:
priced = price_optimizer(
    PowerPriceResponse(elasticity={"Social": 0.4}, assume_delivery_units=True)
)
constant_price_plan = price_optimizer(None).allocate_budget(total_budget=1_000).budgets
priced_plan = priced.allocate_budget(total_budget=1_000).budgets

gain = priced.evaluate_response_distribution(
    priced_plan
) - priced.evaluate_response_distribution(constant_price_plan)
print(f"Mean gain of the price-aware plan: {float(gain.mean()):,.1f}")
print(f"Probability the gain is positive:  {float((gain > 0).mean()):.2f}")

Mean gain of the price-aware plan: 602.8
Probability the gain is positive:  0.81


The deprecated `BudgetOptimizerWrapper.sample_response_distribution` takes an allocation in model units, so it needs `result.implied_delivery.mean("date")`, not `result.budgets`. That average matches what the solver used only when delivery is constant over the window: a uniform `budget_distribution_over_period` and a `cost_per_unit` that does not vary by date.

### What `PowerPriceResponse` does not model

- **Volume discounts.** $\gamma$ models bid-up on the extra money of this plan. A price that falls with volume ($\gamma < 0$) makes the problem non-convex, so it is refused. A negotiated discount belongs in `cost_per_unit`.
- **Rate schedules.** One smooth curve cannot hold a committed tranche at a contracted rate while extra money clears at another rate. Fitted to that case, the curve is exact only at the anchor: its marginal price keeps climbing where the true rate is flat, and allocation follows the marginal price.
- **Mixed references.** A `reference_spend` may cover only some channels. Channels it leaves out keep their derived reference. Here only Social is priced, so this case does not come up.

In [23]:
try:
    PowerPriceResponse(elasticity={"Social": -0.1}, assume_delivery_units=True)
except ValueError as error:
    print(error)

1 validation error for PowerPriceResponse
  Value error, PowerPriceResponse: requires 0 <= elasticity < 1, got [-0.1]. At 1 delivery is constant in spend; above it more money buys less and the optimizer drives the channel to its lower bound. Below 0 the price falls with volume, which makes delivery convex: a negotiated or contract discount belongs in cost_per_unit, the base price. [type=value_error, input_value={'elasticity': {'Social':...e_delivery_units': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/value_error


---

## Key Takeaways

1. **Choose the unit that best represents the causal input** — impressions,
   clicks, GRPs, or even spend — depending on data quality and modelling
   context. There is no one-size-fits-all rule; it is the modeller's call.
2. **Set `cost_per_unit`** to bring every channel onto a common currency
   (dollars, euros, etc.) for fair ROAS comparisons and budget planning.
   This also handles cross-currency channels (e.g. USD vs EUR).
3. **Raw ROAS can be misleading** when channels use different units — always
   check whether you're comparing apples to apples
4. **Budget optimization respects `cost_per_unit`** — the optimizer works in
   the common currency internally and converts to channel-native units before
   evaluating the response function
5. **Sensitivity analysis** with different rates lets you plan for changing
   media prices or exchange rates without refitting the model.
6. When media *prices* trend over the sample and spend is the model input,
   adjust or replace that input *before* `fit`. See {ref}`mmm_media_inflation`.
   `cost_per_unit` does not rewrite the media input.
7. When the price per unit rises with the spend, `PowerPriceResponse` prices it.
   It needs a fit in delivery units and an explicit `assume_delivery_units=True`.
   Read `implied_price` on every channel and sweep the elasticity before trusting
   the plan. See {ref}`mmm_cost_per_unit_price_response`.

In [24]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pymc_marketing

Last updated: Thu, 12 Nov 2026

Python implementation: CPython
Python version       : 3.13.12
IPython version      : 9.18.1

pymc_marketing: 1.4.0

arviz         : 1.3.0
numpy         : 2.5.4
pandas        : 3.1.1
plotly        : 7.1.0
pymc_marketing: 1.4.0
xarray        : 2026.9.0

Watermark: 2.6.0

